In [1]:
import pandas as pd
import re

INPUT_FILE = "TextToGloss.xlsx"
SHEET_NAME = 0

TEXT_COL = "text_ar"
GLOSS_COL = "gloss_ar"

df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(10)

Shape: (128, 3)
Columns: ['id', 'text_ar', 'gloss_ar']


,id,text_ar,gloss_ar
0,1,انتبه,انتبه خطر تمام
1,2,حبة 3 مرات قبل الأكل بنص ساعة.,3 مرات أكل قبل نص ساعة حبة
2,3,حبة 3 مرات قبل الأكل.,3 مرات أكل قبل حبة
3,4,حبة 3 مرات قبل الأكل بساعة.,3 مرات أكل قبل ساعة حبة
4,5,معلقة كبيرة 3 مرات .,3 مرة ملعقة-كبيرة شراب تمام
5,6,6 شهور,6 شهر
6,7,نقطتين بكل طرف,اذن يمين نقطة-نقطة اذن يسار نقطة-نقطة
7,8,لمدة تلات أيام,استمرار ثلاثة يوم بس تمام
8,9,خده ورا الأكل,اكل بعد حبة تمام معدة فاضية لا
9,10,بعد الأكل,اكل خلص فورا دواء تمام


In [2]:
df = df.dropna(subset=[TEXT_COL, GLOSS_COL]).copy()

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[GLOSS_COL] = df[GLOSS_COL].astype(str).str.strip()

df = df[(df[TEXT_COL] != "") & (df[GLOSS_COL] != "")].copy()

print("Shape after dropping empty rows:", df.shape)
df.head(10)

Shape after dropping empty rows: (128, 3)


,id,text_ar,gloss_ar
0,1,انتبه,انتبه خطر تمام
1,2,حبة 3 مرات قبل الأكل بنص ساعة.,3 مرات أكل قبل نص ساعة حبة
2,3,حبة 3 مرات قبل الأكل.,3 مرات أكل قبل حبة
3,4,حبة 3 مرات قبل الأكل بساعة.,3 مرات أكل قبل ساعة حبة
4,5,معلقة كبيرة 3 مرات .,3 مرة ملعقة-كبيرة شراب تمام
5,6,6 شهور,6 شهر
6,7,نقطتين بكل طرف,اذن يمين نقطة-نقطة اذن يسار نقطة-نقطة
7,8,لمدة تلات أيام,استمرار ثلاثة يوم بس تمام
8,9,خده ورا الأكل,اكل بعد حبة تمام معدة فاضية لا
9,10,بعد الأكل,اكل خلص فورا دواء تمام


In [3]:
def normalize_ar_basic(text: str) -> str:
    text = str(text).strip()

    # إزالة التشكيل
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)

    # توحيد الحروف
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)

    # توحيد علامات عربية/إنكليزية
    text = text.replace("،", ",")
    text = text.replace("؛", ";")
    text = text.replace("؟", "?")

    # إزالة التطويل
    text = text.replace("ـ", "")

    # توحيد المسافات
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_punctuation(text: str) -> str:
    text = str(text)

    # فصل بعض علامات الترقيم
    text = re.sub(r"([,;:.!?()\[\]{}])", r" \1 ", text)

    # توحيد الشرطة
    text = re.sub(r"\s*-\s*", " - ", text)

    # توحيد المسافات
    text = re.sub(r"\s+", " ", text).strip()

    return text


def remove_extra_punct(text: str) -> str:
    text = str(text)

    text = re.sub(r"!{2,}", "!", text)
    text = re.sub(r"\?{2,}", "?", text)
    text = re.sub(r"\.{2,}", ".", text)
    text = re.sub(r",{2,}", ",", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

In [4]:
DUAL_EXCEPTIONS = {
    "مرتين", "مرتان",
    "حبتين",
    "كبسولتين",
    "قطرتين",
    "نقطتين",
    "يومين",
    "ساعتين",
    "اسبوعين", "أسبوعين"
}

PLURAL_TO_SINGULAR = {
    # زمن / عدد
    "ايام": "يوم",
    "أيام": "يوم",
    "ساعات": "ساعة",
    "دقائق": "دقيقة",
    "دقايق": "دقيقة",
    "اسابيع": "اسبوع",
    "أسابيع": "اسبوع",
    "شهور": "شهر",
    "اشهر": "شهر",

    # أكل / وجبات
    "وجبات": "وجبة",

    # جرعات / أشكال دوائية
    "جرعات": "جرعة",
    "اقراص": "قرص",
    "أقراص": "قرص",
    "كبسولات": "كبسولة",
    "قطرات": "قطرة",
    "نقاط": "نقطة",

    # هذه اختيارات domain-specific
    "حبوب": "حبة",
    "ادوية": "دواء",
    "أدوية": "دواء",
    "مراهم": "مرهم",
    "كريمات": "كريم",
    "اشربة": "شراب",
    "أشربة": "شراب",

    # أعراض وأعضاء
    # إذا ما بدك "اعراض -> عرض" احذفي السطرين التاليين
    "اعراض": "عرض",
    "أعراض": "عرض",
    "عيون": "عين",
    "اذان": "اذن",
    "آذان": "اذن"
}

In [5]:
NORMALIZED_DUAL_EXCEPTIONS = {normalize_ar_basic(x) for x in DUAL_EXCEPTIONS}
NORMALIZED_PLURAL_MAP = {
    normalize_ar_basic(k): normalize_ar_basic(v)
    for k, v in PLURAL_TO_SINGULAR.items()
}

def singularize_token(token: str) -> str:
    tok = normalize_ar_basic(token)

    # أبقي المثنى كما هو
    if tok in NORMALIZED_DUAL_EXCEPTIONS:
        return tok

    # تحويلات يدوية أولاً
    if tok in NORMALIZED_PLURAL_MAP:
        return NORMALIZED_PLURAL_MAP[tok]

    # جمع مؤنث سالم: وجبات -> وجبة
    if len(tok) > 3 and tok.endswith("ات"):
        return tok[:-2] + "ة"

    # جمع مذكر سالم: مستخدمون / مستخدمين -> مستخدم
    if len(tok) > 4 and (tok.endswith("ون") or tok.endswith("ين")):
        base = tok[:-2]

        # لا نكسر المثنى أو أشياء حساسة
        blocked_bases = {
            "مرت", "حبت", "كبسولت", "قطرت", "نقطت",
            "يوم", "ساعت", "اسبوع", "أسبوع"
        }
        if base not in blocked_bases:
            return base

    return tok


def singularize_text(text: str) -> str:
    parts = str(text).split()
    new_parts = []

    for part in parts:
        m = re.match(r"^([^\w\u0600-\u06FF]*)([\w\u0600-\u06FF]+)([^\w\u0600-\u06FF]*)$", part)
        if m:
            prefix, core, suffix = m.groups()
            core_new = singularize_token(core)
            new_parts.append(prefix + core_new + suffix)
        else:
            new_parts.append(part)

    return " ".join(new_parts)

In [6]:
def normalize_text_ar(text: str) -> str:
    text = normalize_ar_basic(text)
    text = singularize_text(text)
    text = remove_extra_punct(text)
    text = normalize_punctuation(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_gloss_ar(gloss: str) -> str:
    gloss = normalize_ar_basic(gloss)
    gloss = singularize_text(gloss)
    gloss = remove_extra_punct(gloss)
    gloss = normalize_punctuation(gloss)
    gloss = re.sub(r"\s+", " ", gloss).strip()
    return gloss

In [7]:
df["text_ar_norm"] = df[TEXT_COL].apply(normalize_text_ar)
df["gloss_ar_norm"] = df[GLOSS_COL].apply(normalize_gloss_ar)

df[[TEXT_COL, "text_ar_norm", GLOSS_COL, "gloss_ar_norm"]].head(20)

,text_ar,text_ar_norm,gloss_ar,gloss_ar_norm
0,انتبه,انتبه,انتبه خطر تمام,انتبه خطر تمام
1,حبة 3 مرات قبل الأكل بنص ساعة.,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرات أكل قبل نص ساعة حبة,3 مرة اكل قبل نص ساعة حبة
2,حبة 3 مرات قبل الأكل.,حبة 3 مرة قبل الاكل .,3 مرات أكل قبل حبة,3 مرة اكل قبل حبة
3,حبة 3 مرات قبل الأكل بساعة.,حبة 3 مرة قبل الاكل بساعة .,3 مرات أكل قبل ساعة حبة,3 مرة اكل قبل ساعة حبة
4,معلقة كبيرة 3 مرات .,معلقة كبيرة 3 مرة .,3 مرة ملعقة-كبيرة شراب تمام,3 مرة ملعقة - كبيرة شراب تمام
5,6 شهور,6 شهر,6 شهر,6 شهر
6,نقطتين بكل طرف,نقطتين بكل طرف,اذن يمين نقطة-نقطة اذن يسار نقطة-نقطة,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة
7,لمدة تلات أيام,لمدة تلة يوم,استمرار ثلاثة يوم بس تمام,استمرار ثلاثة يوم بس تمام
8,خده ورا الأكل,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,اكل بعد حبة تمام معدة فاضية لا
9,بعد الأكل,بعد الاكل,اكل خلص فورا دواء تمام,اكل خلص فورا دواء تمام


In [8]:
changed = df[
    (df[TEXT_COL] != df["text_ar_norm"]) |
    (df[GLOSS_COL] != df["gloss_ar_norm"])
].copy()

print("Number of changed rows:", len(changed))
changed[[TEXT_COL, "text_ar_norm", GLOSS_COL, "gloss_ar_norm"]].head(30)

Number of changed rows: 102


,text_ar,text_ar_norm,gloss_ar,gloss_ar_norm
1,حبة 3 مرات قبل الأكل بنص ساعة.,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرات أكل قبل نص ساعة حبة,3 مرة اكل قبل نص ساعة حبة
2,حبة 3 مرات قبل الأكل.,حبة 3 مرة قبل الاكل .,3 مرات أكل قبل حبة,3 مرة اكل قبل حبة
3,حبة 3 مرات قبل الأكل بساعة.,حبة 3 مرة قبل الاكل بساعة .,3 مرات أكل قبل ساعة حبة,3 مرة اكل قبل ساعة حبة
4,معلقة كبيرة 3 مرات .,معلقة كبيرة 3 مرة .,3 مرة ملعقة-كبيرة شراب تمام,3 مرة ملعقة - كبيرة شراب تمام
5,6 شهور,6 شهر,6 شهر,6 شهر
6,نقطتين بكل طرف,نقطتين بكل طرف,اذن يمين نقطة-نقطة اذن يسار نقطة-نقطة,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة
7,لمدة تلات أيام,لمدة تلة يوم,استمرار ثلاثة يوم بس تمام,استمرار ثلاثة يوم بس تمام
8,خده ورا الأكل,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,اكل بعد حبة تمام معدة فاضية لا
9,بعد الأكل,بعد الاكل,اكل خلص فورا دواء تمام,اكل خلص فورا دواء تمام
10,مع أكلة دسمة.,مع اكلة دسمة .,اكل دسم ثقيل لازم تمام بيض حليب زيت تمام دواء ...,اكل دسم ثقيل لازم تمام بيض حليب زيت تمام دواء ...


In [9]:
before_dedup = len(df)

df_clean = df.drop_duplicates(subset=["text_ar_norm", "gloss_ar_norm"]).copy()

after_dedup = len(df_clean)

print("Rows before dedup:", before_dedup)
print("Rows after dedup :", after_dedup)
print("Removed duplicates:", before_dedup - after_dedup)

Rows before dedup: 128
Rows after dedup : 128
Removed duplicates: 0


In [10]:
df_final = df_clean.copy()

df_final["text_ar"] = df_final["text_ar_norm"]
df_final["gloss_ar"] = df_final["gloss_ar_norm"]

# حذف الأعمدة المؤقتة
df_final = df_final.drop(columns=["text_ar_norm", "gloss_ar_norm"])

# إعادة ترتيب الأعمدة لو حبيتي
preferred_cols = []
for c in ["text_ar", "gloss_ar"]:
    if c in df_final.columns:
        preferred_cols.append(c)

other_cols = [c for c in df_final.columns if c not in preferred_cols]
df_final = df_final[preferred_cols + other_cols]

df_final.head(20)

,text_ar,gloss_ar,id
0,انتبه,انتبه خطر تمام,1
1,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,2
2,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,3
3,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,4
4,معلقة كبيرة 3 مرة .,3 مرة ملعقة - كبيرة شراب تمام,5
5,6 شهر,6 شهر,6
6,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,7
7,لمدة تلة يوم,استمرار ثلاثة يوم بس تمام,8
8,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,9
9,بعد الاكل,اكل خلص فورا دواء تمام,10


In [11]:
OUTPUT_FILE = "TextToGloss_normalized.xlsx"
df_final.to_excel(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Final shape:", df_final.shape)

Saved: TextToGloss_normalized.xlsx
Final shape: (128, 3)


In [12]:
df_final.sample(min(20, len(df_final)), random_state=42)

,text_ar,gloss_ar,id
55,ويفضل تبعد عن الشغلة الباردة,شرب بارد لا - ممنوع,56
40,"دير بالك من الضوء , خليه بكرتونته دايما .",دواء داخل كرتونة تمام ضوء شمس لا - خطر,41
19,عند الضرورة .,جسم تعب ضرورة بس دواء,20
31,بخاخ مرتين باليوم,دواء انف بخ تمام يمين واحد يسار واحد تمام تكرا...,32
98,استمر بالتنفس بعمق حتي ينتهي المحلول تماما .,نفس عميق استمرار دواء كلو خلص تمام,99
56,حصرا الصبح,صباح بكير حبة تمام لازم,57
69,"حبة بالنهار بعد الاكل لمدة تلة يوم ,",كل يوم حبة اكل بعد تمام استمرار ثلاثة يوم بس,70
104,لا تخلط هاد النوع مع دواء الاعشاب,دواء اعشاب خلط لا - ممنوع,105
81,عدم لمس طرف القطارة للع,دواء عين طرف لمس ممنوع خطر,82
26,لازم تقطعيه,دواء تمام وقف - فورا,27
